# Paper Tables

This notebook builds the paper-facing tables from `results/summary/`.

Recommended placement:

- Main paper: native retrieval and pairwise conversion
- Appendix: shared-retrieval context and 3-way conversion


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

SUMMARY_DIR = ROOT / "results" / "summary"
TABLE_DIR = ROOT / "results" / "paper" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

retrieval_summary = pd.read_csv(SUMMARY_DIR / "retrieval_summary.csv")
conversion_summary = pd.read_csv(SUMMARY_DIR / "conversion_summary.csv")


In [2]:
POOL_LABELS = {
    "none": "Native",
    "shared-eeg-meg": "EEG↔MEG",
    "shared-eeg-fmri": "EEG↔fMRI",
    "shared-fmri-meg": "MEG↔fMRI",
    "shared-eeg-fmri-meg": "EEG↔MEG↔fMRI",
}
MODALITY_ORDER = {"eeg": 0, "meg": 1, "fmri": 2}

def pct(mean, std):
    return f"{mean:.2f} ± {std:.2f}"

def ratio(mean, std):
    return f"{mean:.3f} ± {std:.3f}"

def ratio_only(mean):
    return f"{mean:.3f}"

def pool_label(shared_group):
    return POOL_LABELS.get(shared_group, shared_group)

def save_table(df, stem):
    csv_path = TABLE_DIR / f"{stem}.csv"
    tex_path = TABLE_DIR / f"{stem}.tex"
    df.to_csv(csv_path, index=False)
    tex_path.write_text(df.to_latex(index=False, escape=False))
    print(f"Saved {csv_path}")
    print(f"Saved {tex_path}")
    return df

def build_native_retrieval_table():
    df = retrieval_summary.query("evaluation_scope == 'full' and shared_only == False").copy()
    df = df.sort_values("modality", key=lambda col: col.map(MODALITY_ORDER))
    return pd.DataFrame(
        {
            "Modality": df["modality"].str.upper(),
            "Candidate Set": df["retrieval_dataset_size_mean"].round(0).astype(int).astype(str) + "-way",
            "Top-1 (%)": [pct(m, s) for m, s in zip(df["m2i_top1_mean"], df["m2i_top1_std"])],
            "Top-5 (%)": [pct(m, s) for m, s in zip(df["m2i_top5_mean"], df["m2i_top5_std"])],
            "CLIP 2-way (%)": [pct(m, s) for m, s in zip(df["m2i_two_way_mean"], df["m2i_two_way_std"])],
        }
    )

def build_shared_retrieval_context_table():
    shared_df = retrieval_summary.query("shared_only == True").copy()
    full_lookup = {
        row["modality"]: row["m2i_two_way_mean"]
        for _, row in retrieval_summary.query("evaluation_scope == 'full' and shared_only == False").iterrows()
    }
    shared_df["retained_two_way"] = shared_df.apply(
        lambda row: row["m2i_two_way_mean"] / full_lookup[row["modality"]],
        axis=1,
    )
    shared_df = shared_df.sort_values(
        ["modality", "evaluation_scope", "shared_group"],
        key=lambda col: col.map(MODALITY_ORDER) if col.name == "modality" else col,
    )
    return pd.DataFrame(
        {
            "Modality": shared_df["modality"].str.upper(),
            "Scope": shared_df["evaluation_scope"].str.replace("_", " ").str.title(),
            "Shared Pool": shared_df["shared_group"].map(pool_label),
            "Top-1 (%)": [pct(m, s) for m, s in zip(shared_df["m2i_top1_mean"], shared_df["m2i_top1_std"])],
            "CLIP 2-way (%)": [pct(m, s) for m, s in zip(shared_df["m2i_two_way_mean"], shared_df["m2i_two_way_std"])],
            "2-way / Native": [ratio_only(m) for m in shared_df["retained_two_way"]],
        }
    )

def build_conversion_table(scope):
    df = conversion_summary.query("evaluation_scope == @scope").copy()
    df = df.sort_values(["shared_group", "source_modality", "target_modality"])
    rows = []
    for _, row in df.iterrows():
        rows.append(
            {
                "Shared Pool": pool_label(row["shared_group"]),
                "Direction": f"{row['source_modality'].upper()}→{row['target_modality'].upper()}",
                "Top-1 (%)": pct(row["forward_top1_mean"], row["forward_top1_std"]),
                "Top-5 (%)": pct(row["forward_top5_mean"], row["forward_top5_std"]),
                "CLIP 2-way (%)": pct(row["forward_two_way_mean"], row["forward_two_way_std"]),
                "2-way / matched retrieval": ratio(
                    row["forward_normalized_two_way_mean"],
                    row["forward_normalized_two_way_std"],
                ),
                "2-way / native retrieval": ratio(
                    row["forward_normalized_two_way_full_mean"],
                    row["forward_normalized_two_way_full_std"],
                ),
            }
        )
        rows.append(
            {
                "Shared Pool": pool_label(row["shared_group"]),
                "Direction": f"{row['target_modality'].upper()}→{row['source_modality'].upper()}",
                "Top-1 (%)": pct(row["reverse_top1_mean"], row["reverse_top1_std"]),
                "Top-5 (%)": pct(row["reverse_top5_mean"], row["reverse_top5_std"]),
                "CLIP 2-way (%)": pct(row["reverse_two_way_mean"], row["reverse_two_way_std"]),
                "2-way / matched retrieval": ratio(
                    row["reverse_normalized_two_way_mean"],
                    row["reverse_normalized_two_way_std"],
                ),
                "2-way / native retrieval": ratio(
                    row["reverse_normalized_two_way_full_mean"],
                    row["reverse_normalized_two_way_full_std"],
                ),
            }
        )
    return pd.DataFrame(rows)


## Main-Paper Tables

These are the most compact tables for the nine-page NeurIPS main paper:

- native retrieval performance on each modality's native evaluation protocol
- pairwise shared conversion using matched-scope normalization


In [3]:
native_retrieval_table = save_table(build_native_retrieval_table(), "table1_native_retrieval")
display(native_retrieval_table)


Saved /Users/thomas/Documents/Projects/brainalign_ext_meg_fmri/results/paper/tables/table1_native_retrieval.csv
Saved /Users/thomas/Documents/Projects/brainalign_ext_meg_fmri/results/paper/tables/table1_native_retrieval.tex


,Modality,Candidate Set,Top-1 (%),Top-5 (%),CLIP 2-way (%)
0,EEG,200-way,12.95 ± 3.29,39.35 ± 7.70,90.57 ± 2.12
8,MEG,200-way,8.00 ± 4.08,25.12 ± 11.04,82.95 ± 5.24
4,FMRI,100-way,33.33 ± 6.60,67.67 ± 5.44,93.87 ± 1.43


In [4]:
pairwise_conversion_table = save_table(build_conversion_table("pair"), "table2_pairwise_conversion")
display(pairwise_conversion_table)


Saved /Users/thomas/Documents/Projects/brainalign_ext_meg_fmri/results/paper/tables/table2_pairwise_conversion.csv
Saved /Users/thomas/Documents/Projects/brainalign_ext_meg_fmri/results/paper/tables/table2_pairwise_conversion.tex


,Shared Pool,Direction,Top-1 (%),Top-5 (%),CLIP 2-way (%),2-way / matched retrieval,2-way / native retrieval
0,EEG↔fMRI,EEG→FMRI,1.52 ± 1.10,5.93 ± 2.05,61.69 ± 4.79,0.868 ± 0.068,0.657 ± 0.049
1,EEG↔fMRI,FMRI→EEG,1.27 ± 0.92,5.95 ± 2.52,61.58 ± 4.92,0.901 ± 0.072,0.680 ± 0.051
2,EEG↔MEG,EEG→MEG,2.92 ± 1.49,12.72 ± 4.33,72.89 ± 5.96,0.924 ± 0.030,0.878 ± 0.033
3,EEG↔MEG,MEG→EEG,3.19 ± 1.73,12.60 ± 4.91,73.41 ± 5.77,0.842 ± 0.064,0.811 ± 0.063
4,MEG↔fMRI,MEG→FMRI,1.00 ± 0.74,5.17 ± 2.55,60.02 ± 5.52,0.807 ± 0.077,0.639 ± 0.059
5,MEG↔fMRI,FMRI→MEG,1.21 ± 0.90,6.29 ± 3.31,60.27 ± 5.77,0.925 ± 0.045,0.725 ± 0.031


## Appendix Tables

These tables help interpret the main results but are better suited for the appendix:

- how much each shared pool degrades retrieval relative to the native setting
- three-way shared conversion, which is a stronger stress test than the pairwise main benchmark


In [5]:
shared_retrieval_context_table = save_table(
    build_shared_retrieval_context_table(),
    "tableA1_shared_retrieval_context",
)
display(shared_retrieval_context_table)


Saved /Users/thomas/Documents/Projects/brainalign_ext_meg_fmri/results/paper/tables/tableA1_shared_retrieval_context.csv
Saved /Users/thomas/Documents/Projects/brainalign_ext_meg_fmri/results/paper/tables/tableA1_shared_retrieval_context.tex


,Modality,Scope,Shared Pool,Top-1 (%),CLIP 2-way (%),2-way / Native
1,EEG,Pair,EEG↔fMRI,2.95 ± 1.72,68.98 ± 9.24,0.762
2,EEG,Pair,EEG↔MEG,11.00 ± 3.67,87.18 ± 2.61,0.963
3,EEG,Three Way,EEG↔MEG↔fMRI,2.75 ± 1.75,68.44 ± 10.60,0.756
9,MEG,Pair,EEG↔MEG,6.88 ± 2.99,78.95 ± 7.17,0.952
10,MEG,Pair,MEG↔fMRI,2.88 ± 1.56,65.47 ± 8.16,0.789
11,MEG,Three Way,EEG↔MEG↔fMRI,1.88 ± 1.47,59.91 ± 4.84,0.722
5,FMRI,Pair,EEG↔fMRI,4.17 ± 1.31,71.15 ± 2.51,0.758
6,FMRI,Pair,MEG↔fMRI,2.83 ± 0.85,74.49 ± 2.85,0.793
7,FMRI,Three Way,EEG↔MEG↔fMRI,2.17 ± 1.43,71.47 ± 2.40,0.761


In [6]:
three_way_conversion_table = save_table(
    build_conversion_table("three_way"),
    "tableA2_three_way_conversion",
)
display(three_way_conversion_table)


Saved /Users/thomas/Documents/Projects/brainalign_ext_meg_fmri/results/paper/tables/tableA2_three_way_conversion.csv
Saved /Users/thomas/Documents/Projects/brainalign_ext_meg_fmri/results/paper/tables/tableA2_three_way_conversion.tex


,Shared Pool,Direction,Top-1 (%),Top-5 (%),CLIP 2-way (%),2-way / matched retrieval,2-way / native retrieval
0,EEG↔MEG↔fMRI,EEG→FMRI,1.43 ± 0.93,5.87 ± 2.46,60.98 ± 5.31,0.854 ± 0.076,0.650 ± 0.056
1,EEG↔MEG↔fMRI,FMRI→EEG,1.32 ± 0.98,6.35 ± 2.70,60.97 ± 5.43,0.902 ± 0.080,0.673 ± 0.059
2,EEG↔MEG↔fMRI,EEG→MEG,0.81 ± 0.73,4.04 ± 1.96,57.23 ± 6.75,0.955 ± 0.068,0.689 ± 0.057
3,EEG↔MEG↔fMRI,MEG→EEG,0.81 ± 0.84,3.96 ± 2.07,57.14 ± 6.74,0.849 ± 0.126,0.631 ± 0.074
4,EEG↔MEG↔fMRI,MEG→FMRI,0.88 ± 0.54,3.88 ± 1.23,55.61 ± 3.64,0.779 ± 0.057,0.593 ± 0.039
5,EEG↔MEG↔fMRI,FMRI→MEG,0.71 ± 0.59,3.92 ± 1.48,55.85 ± 3.33,0.935 ± 0.044,0.674 ± 0.032
